# Testing & Training — Pipeline Huấn Luyện YOLO11n + CBAM (Dataset V2)

> **Lưu ý quan trọng trước khi chạy:**
> 1. **Phần cứng Runtime:** Ưu tiên bật **GPU** (T4, V100, A100 tại menu `Runtime` -> `Change runtime type`).
> 2. **Lối tắt Drive:** Tạo lối tắt (Shortcut) thư mục chung `KLTN-2026-testingNtraining` (hoặc `KLTN_38`) vào `My Drive`.
> 3. **Quy chuẩn Augmentation Online (`v2`):** Mọi thí nghiệm tự động áp dụng hồ sơ `v2` (Mosaic 0.5, close_mosaic 10, hsv_v 0.15, translate 0.05, scale 0.1, fliplr 0.0, augmentations=[]) tuân thủ đúng `docs/protocols/aug_data.md`.
> 4. **Cấu trúc lưu trữ đồng bộ:** Kết quả được lưu tại `{RUNS_DIR}/{variant}` với tên chuẩn: `baseline`, `cbam_backbone`, `cbam_neck`, `cbam_head`, `cbam_neck_head`, `cbam_backbone_head`.
> 5. **Khôi phục (Resume):** Mỗi mô hình đều có cell resume riêng biệt kèm theo nếu phiên Colab bị ngắt kết nối.

## Bước 1️⃣ & 2️⃣: Thiết Lập Môi Trường, Google Drive & Chuẩn Bị Dữ Liệu V2

In [ ]:
import hashlib
import os
import sys
import yaml
from pathlib import Path
from google.colab import drive

SELECTED_RATIO = "70_15_15"

drive.mount('/content/drive')

CANDIDATES = [
    "/content/drive/MyDrive/KLTN-2026-testingNtraining",
    "/content/drive/MyDrive/KLTN_38/KLTN-2026-testingNtraining",
    "/content/drive/MyDrive/KLTN_38 - Thầy Bảo/KLTN-2026-testingNtraining",
    "/content/drive/MyDrive/KLTN_38"
]

SHARED_DRIVE = next((p for p in CANDIDATES if os.path.exists(p)), None)
if SHARED_DRIVE is None:
    found = list(Path("/content/drive/MyDrive").rglob("dataset-v2"))
    if found:
        SHARED_DRIVE = str(found[0].parent.parent)

if SHARED_DRIVE is None:
    raise FileNotFoundError(
        "❌ CHƯA TÌM THẤY THƯ MỤC CHUNG!\n"
        "👉 Hãy vào Google Drive web -> Chuột phải vào thư mục chung -> 'Thêm lối tắt vào Drive' -> chọn 'My Drive' rồi chạy lại cell này."
    )

print(f"✅ Thư mục làm việc chung trên Drive: {SHARED_DRIVE}")

!rm -rf /content/klcn2026
!git clone https://github.com/mizzhau/yolo11n-cbam-mvtec-defect-detection.git /content/klcn2026
%cd /content/klcn2026
!git checkout main
!git pull origin main

!pip install -q ultralytics==8.3.253 albumentations==2.0.8 tabulate pyyaml
import ultralytics
assert ultralytics.__version__ == "8.3.253", f"Phiên bản Ultralytics không khớp: {ultralytics.__version__}"

RUNS_DIR = f"{SHARED_DRIVE}/runs_exp2_{SELECTED_RATIO}"
for sub in ["baseline", "cbam_backbone", "cbam_neck", "cbam_head", "cbam_neck_head", "cbam_backbone_head"]:
    os.makedirs(f"{RUNS_DIR}/{sub}", exist_ok=True)
print(f"✅ Thư mục lưu trữ kết quả trên Drive: {RUNS_DIR}")

zip_hits = list(Path(SHARED_DRIVE).rglob(f"KLTN_prep_v2_{SELECTED_RATIO}.zip"))
if not zip_hits:
    raise FileNotFoundError(f"❌ Không tìm thấy file zip: KLTN_prep_v2_{SELECTED_RATIO}.zip trong {SHARED_DRIVE}")
ZIP_SRC = str(zip_hits[0])

def sha256(p, chunk=1<<24):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()

shap = Path(ZIP_SRC + ".sha256")
if shap.exists():
    exp_hash = shap.read_text(encoding="utf-8").split()[0].lower()
    act_hash = sha256(ZIP_SRC).lower()
    assert act_hash == exp_hash, f"❌ SHA-256 checksum lệch! Kỳ vọng: {exp_hash}, thực tế: {act_hash}"
    print(f"✅ SHA-256 Checksum khớp 100%: {act_hash[:16]}...")

EXTRACT_DIR = f"/content/dataset_v2_{SELECTED_RATIO}"
!mkdir -p "{EXTRACT_DIR}"
!unzip -q -o "{ZIP_SRC}" -d "{EXTRACT_DIR}"
print(f"✅ Giải nén hoàn tất vào: {EXTRACT_DIR}")

DATASET_ROOT = Path(EXTRACT_DIR) / "KLTN_prep_v2" if (Path(EXTRACT_DIR) / "KLTN_prep_v2").exists() else Path(EXTRACT_DIR)
ZIP_DATA_YAML = DATASET_ROOT / f"data_{SELECTED_RATIO}.yaml"
assert ZIP_DATA_YAML.exists(), f"❌ Không tìm thấy {ZIP_DATA_YAML}"

with open(ZIP_DATA_YAML, "r", encoding="utf-8") as f:
    ref_data = yaml.safe_load(f)
class_names = ref_data.get("names", [])
assert ref_data.get("nc", len(class_names)) == 48 and len(class_names) == 48, "❌ Cảnh báo số class không đúng 48!"

colab_yaml_content = {
    "path": str(DATASET_ROOT),
    "train": str(DATASET_ROOT / f"train_{SELECTED_RATIO}.txt"),
    "val": str(DATASET_ROOT / f"val_{SELECTED_RATIO}.txt"),
    "test": str(DATASET_ROOT / f"test_{SELECTED_RATIO}.txt"),
    "nc": 48,
    "names": class_names
}

ACTIVE_YAML_PATH = "/content/klcn2026/configs/data/exp_2/data_colab_active.yaml"
os.makedirs(os.path.dirname(ACTIVE_YAML_PATH), exist_ok=True)
with open(ACTIVE_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(colab_yaml_content, f, allow_unicode=True, sort_keys=False)

print(f"✅ Đã tạo cấu hình YOLO tại: {ACTIVE_YAML_PATH}")
print("\n🚀 Toàn bộ môi trường và dữ liệu V2 đã sẵn sàng huấn luyện!")


## Bước 3️⃣: Thực Nghiệm Huấn Luyện


> Lưu Ý: Các tự động hoàn tất sau khi huấn luyện:
> - Lưu trọng số `best.pt` & `last.pt`
> - Đo đạc Latency ms, FPS, Params M & F1-Score trên test set 
> - Trích xuất bộ 3 trực quan hóa (Grad-CAM, Side-by-Side, Failure Cases) chỉ dùng test set

### Cell 3A: Huấn Luyện & Đánh Giá BASELINE (YOLO11n - Exp 0) 🌟


In [ ]:
!python src/training/train_baseline.py \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --aug_profile v2 \
    --project "{RUNS_DIR}" \
    --name baseline

import os
from pathlib import Path

WEIGHTS_PT = f"{RUNS_DIR}/baseline/weights/best.pt"
TEST_FILE = f"{DATASET_ROOT}/test_{SELECTED_RATIO}.txt"
VISUAL_DIR = f"{RUNS_DIR}/visual_eval_baseline"
BENCHMARK_OUT = f"{RUNS_DIR}/baseline/benchmark_latency.md"

if os.path.exists(WEIGHTS_PT):
    !python src/evaluation/benchmark_latency.py \
        --weights "{WEIGHTS_PT}" \
        --data configs/data/exp_2/data_colab_active.yaml \
        --imgsz 640 \
        --batch 1 \
        --warmup 10 \
        --runs 100 \
        --out "{BENCHMARK_OUT}"

    !python src/inference/visual_analysis.py \
        --weights "{WEIGHTS_PT}" \
        --test_file "{TEST_FILE}" \
        --output_dir "{VISUAL_DIR}" \
        --imgsz 640 \
        --max_images 30
else:
    print(f"⚠️ Chưa có {WEIGHTS_PT} để thực hiện đánh giá và trực quan hóa.")


#### Cell 3A_crack: Chạy Tiếp Tục (Resume) BASELINE (YOLO11n - Exp 0) Nếu Bị Crash 🔄


In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

CHECKPOINT = f"{RUNS_DIR}/baseline/weights/last.pt"
WEIGHTS_PT = f"{RUNS_DIR}/baseline/weights/best.pt"
TEST_FILE = f"{DATASET_ROOT}/test_{SELECTED_RATIO}.txt"
VISUAL_DIR = f"{RUNS_DIR}/visual_eval_baseline"
BENCHMARK_OUT = f"{RUNS_DIR}/baseline/benchmark_latency.md"

if os.path.exists(CHECKPOINT):
    print(f"🔄 Đang tiếp tục huấn luyện BASELINE (YOLO11n - Exp 0) từ: {CHECKPOINT}")
    model = YOLO(CHECKPOINT)
    model.train(resume=True)
    if os.path.exists(WEIGHTS_PT):
        !python src/evaluation/benchmark_latency.py \
            --weights "{WEIGHTS_PT}" \
            --data configs/data/exp_2/data_colab_active.yaml \
            --imgsz 640 \
            --batch 1 \
            --warmup 10 \
            --runs 100 \
            --out "{BENCHMARK_OUT}"

        !python src/inference/visual_analysis.py \
            --weights "{WEIGHTS_PT}" \
            --test_file "{TEST_FILE}" \
            --output_dir "{VISUAL_DIR}" \
            --imgsz 640 \
            --max_images 30
else:
    print(f"❌ Không tìm thấy checkpoint: {CHECKPOINT}")


### Cell 3B: Huấn Luyện & Đánh Giá Tự Động CBAM BACKBONE (Exp 1) 🌟


In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_backbone.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --aug_profile v2 \
    --project "{RUNS_DIR}" \
    --name cbam_backbone

!python src/evaluation/plot_comparison.py \
    --baseline_dir "{RUNS_DIR}/baseline" \
    --cbam_dir "{RUNS_DIR}/cbam_backbone" \
    --output_dir "{RUNS_DIR}/cbam_backbone/evaluation"

import os
from pathlib import Path

WEIGHTS_PT = f"{RUNS_DIR}/cbam_backbone/weights/best.pt"
TEST_FILE = f"{DATASET_ROOT}/test_{SELECTED_RATIO}.txt"
VISUAL_DIR = f"{RUNS_DIR}/visual_eval_cbam_backbone"
BENCHMARK_OUT = f"{RUNS_DIR}/cbam_backbone/benchmark_latency.md"

if os.path.exists(WEIGHTS_PT):
    !python src/evaluation/benchmark_latency.py \
        --weights "{WEIGHTS_PT}" \
        --data configs/data/exp_2/data_colab_active.yaml \
        --imgsz 640 \
        --batch 1 \
        --warmup 10 \
        --runs 100 \
        --out "{BENCHMARK_OUT}"

    !python src/inference/visual_analysis.py \
        --weights "{WEIGHTS_PT}" \
        --test_file "{TEST_FILE}" \
        --output_dir "{VISUAL_DIR}" \
        --imgsz 640 \
        --max_images 30
else:
    print(f"⚠️ Chưa có {WEIGHTS_PT} để thực hiện đánh giá và trực quan hóa.")


#### Cell 3B_crack: Chạy Tiếp Tục (Resume) CBAM BACKBONE (Exp 1) Nếu Bị Crash 🔄


In [ ]:
import os
from pathlib import Path
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()
from ultralytics import YOLO

CHECKPOINT = f"{RUNS_DIR}/cbam_backbone/weights/last.pt"
WEIGHTS_PT = f"{RUNS_DIR}/cbam_backbone/weights/best.pt"
TEST_FILE = f"{DATASET_ROOT}/test_{SELECTED_RATIO}.txt"
VISUAL_DIR = f"{RUNS_DIR}/visual_eval_cbam_backbone"
BENCHMARK_OUT = f"{RUNS_DIR}/cbam_backbone/benchmark_latency.md"

if os.path.exists(CHECKPOINT):
    print(f"🔄 Đang tiếp tục huấn luyện CBAM BACKBONE (Exp 1) từ: {CHECKPOINT}")
    model = YOLO(CHECKPOINT)
    model.train(resume=True)
    !python src/evaluation/plot_comparison.py \
        --baseline_dir "{RUNS_DIR}/baseline" \
        --cbam_dir "{RUNS_DIR}/cbam_backbone" \
        --output_dir "{RUNS_DIR}/cbam_backbone/evaluation"

    if os.path.exists(WEIGHTS_PT):
        !python src/evaluation/benchmark_latency.py \
            --weights "{WEIGHTS_PT}" \
            --data configs/data/exp_2/data_colab_active.yaml \
            --imgsz 640 \
            --batch 1 \
            --warmup 10 \
            --runs 100 \
            --out "{BENCHMARK_OUT}"

        !python src/inference/visual_analysis.py \
            --weights "{WEIGHTS_PT}" \
            --test_file "{TEST_FILE}" \
            --output_dir "{VISUAL_DIR}" \
            --imgsz 640 \
            --max_images 30
else:
    print(f"❌ Không tìm thấy checkpoint: {CHECKPOINT}")


### Cell 3C: Huấn Luyện & Đánh Giá Tự Động CBAM NECK (Exp 2) 🌟


In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_neck.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --aug_profile v2 \
    --project "{RUNS_DIR}" \
    --name cbam_neck

!python src/evaluation/plot_comparison.py \
    --baseline_dir "{RUNS_DIR}/baseline" \
    --cbam_dir "{RUNS_DIR}/cbam_neck" \
    --output_dir "{RUNS_DIR}/cbam_neck/evaluation"

import os
from pathlib import Path

WEIGHTS_PT = f"{RUNS_DIR}/cbam_neck/weights/best.pt"
TEST_FILE = f"{DATASET_ROOT}/test_{SELECTED_RATIO}.txt"
VISUAL_DIR = f"{RUNS_DIR}/visual_eval_cbam_neck"
BENCHMARK_OUT = f"{RUNS_DIR}/cbam_neck/benchmark_latency.md"

if os.path.exists(WEIGHTS_PT):
    !python src/evaluation/benchmark_latency.py \
        --weights "{WEIGHTS_PT}" \
        --data configs/data/exp_2/data_colab_active.yaml \
        --imgsz 640 \
        --batch 1 \
        --warmup 10 \
        --runs 100 \
        --out "{BENCHMARK_OUT}"

    !python src/inference/visual_analysis.py \
        --weights "{WEIGHTS_PT}" \
        --test_file "{TEST_FILE}" \
        --output_dir "{VISUAL_DIR}" \
        --imgsz 640 \
        --max_images 30
else:
    print(f"⚠️ Chưa có {WEIGHTS_PT} để thực hiện đánh giá và trực quan hóa.")


#### Cell 3C_crack: Chạy Tiếp Tục (Resume) CBAM NECK (Exp 2) Nếu Bị Crash 🔄


In [ ]:
import os
from pathlib import Path
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()
from ultralytics import YOLO

CHECKPOINT = f"{RUNS_DIR}/cbam_neck/weights/last.pt"
WEIGHTS_PT = f"{RUNS_DIR}/cbam_neck/weights/best.pt"
TEST_FILE = f"{DATASET_ROOT}/test_{SELECTED_RATIO}.txt"
VISUAL_DIR = f"{RUNS_DIR}/visual_eval_cbam_neck"
BENCHMARK_OUT = f"{RUNS_DIR}/cbam_neck/benchmark_latency.md"

if os.path.exists(CHECKPOINT):
    print(f"🔄 Đang tiếp tục huấn luyện CBAM NECK (Exp 2) từ: {CHECKPOINT}")
    model = YOLO(CHECKPOINT)
    model.train(resume=True)
    !python src/evaluation/plot_comparison.py \
        --baseline_dir "{RUNS_DIR}/baseline" \
        --cbam_dir "{RUNS_DIR}/cbam_neck" \
        --output_dir "{RUNS_DIR}/cbam_neck/evaluation"

    if os.path.exists(WEIGHTS_PT):
        !python src/evaluation/benchmark_latency.py \
            --weights "{WEIGHTS_PT}" \
            --data configs/data/exp_2/data_colab_active.yaml \
            --imgsz 640 \
            --batch 1 \
            --warmup 10 \
            --runs 100 \
            --out "{BENCHMARK_OUT}"

        !python src/inference/visual_analysis.py \
            --weights "{WEIGHTS_PT}" \
            --test_file "{TEST_FILE}" \
            --output_dir "{VISUAL_DIR}" \
            --imgsz 640 \
            --max_images 30
else:
    print(f"❌ Không tìm thấy checkpoint: {CHECKPOINT}")


### Cell 3D: Huấn Luyện & Đánh Giá Tự Động CBAM HEAD (Exp 4) 🌟


In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_head.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --aug_profile v2 \
    --project "{RUNS_DIR}" \
    --name cbam_head

!python src/evaluation/plot_comparison.py \
    --baseline_dir "{RUNS_DIR}/baseline" \
    --cbam_dir "{RUNS_DIR}/cbam_head" \
    --output_dir "{RUNS_DIR}/cbam_head/evaluation"

import os
from pathlib import Path

WEIGHTS_PT = f"{RUNS_DIR}/cbam_head/weights/best.pt"
TEST_FILE = f"{DATASET_ROOT}/test_{SELECTED_RATIO}.txt"
VISUAL_DIR = f"{RUNS_DIR}/visual_eval_cbam_head"
BENCHMARK_OUT = f"{RUNS_DIR}/cbam_head/benchmark_latency.md"

if os.path.exists(WEIGHTS_PT):
    !python src/evaluation/benchmark_latency.py \
        --weights "{WEIGHTS_PT}" \
        --data configs/data/exp_2/data_colab_active.yaml \
        --imgsz 640 \
        --batch 1 \
        --warmup 10 \
        --runs 100 \
        --out "{BENCHMARK_OUT}"

    !python src/inference/visual_analysis.py \
        --weights "{WEIGHTS_PT}" \
        --test_file "{TEST_FILE}" \
        --output_dir "{VISUAL_DIR}" \
        --imgsz 640 \
        --max_images 30
else:
    print(f"⚠️ Chưa có {WEIGHTS_PT} để thực hiện đánh giá và trực quan hóa.")


#### Cell 3D_crack: Chạy Tiếp Tục (Resume) CBAM HEAD (Exp 4) Nếu Bị Crash 🔄


In [ ]:
import os
from pathlib import Path
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()
from ultralytics import YOLO

CHECKPOINT = f"{RUNS_DIR}/cbam_head/weights/last.pt"
WEIGHTS_PT = f"{RUNS_DIR}/cbam_head/weights/best.pt"
TEST_FILE = f"{DATASET_ROOT}/test_{SELECTED_RATIO}.txt"
VISUAL_DIR = f"{RUNS_DIR}/visual_eval_cbam_head"
BENCHMARK_OUT = f"{RUNS_DIR}/cbam_head/benchmark_latency.md"

if os.path.exists(CHECKPOINT):
    print(f"🔄 Đang tiếp tục huấn luyện CBAM HEAD (Exp 4) từ: {CHECKPOINT}")
    model = YOLO(CHECKPOINT)
    model.train(resume=True)
    !python src/evaluation/plot_comparison.py \
        --baseline_dir "{RUNS_DIR}/baseline" \
        --cbam_dir "{RUNS_DIR}/cbam_head" \
        --output_dir "{RUNS_DIR}/cbam_head/evaluation"

    if os.path.exists(WEIGHTS_PT):
        !python src/evaluation/benchmark_latency.py \
            --weights "{WEIGHTS_PT}" \
            --data configs/data/exp_2/data_colab_active.yaml \
            --imgsz 640 \
            --batch 1 \
            --warmup 10 \
            --runs 100 \
            --out "{BENCHMARK_OUT}"

        !python src/inference/visual_analysis.py \
            --weights "{WEIGHTS_PT}" \
            --test_file "{TEST_FILE}" \
            --output_dir "{VISUAL_DIR}" \
            --imgsz 640 \
            --max_images 30
else:
    print(f"❌ Không tìm thấy checkpoint: {CHECKPOINT}")


### Cell 3E: Huấn Luyện & Đánh Giá CBAM NECK + HEAD (Exp 3) 🌟


In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_neck_head.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --aug_profile v2 \
    --project "{RUNS_DIR}" \
    --name cbam_neck_head

!python src/evaluation/plot_comparison.py \
    --baseline_dir "{RUNS_DIR}/baseline" \
    --cbam_dir "{RUNS_DIR}/cbam_neck_head" \
    --output_dir "{RUNS_DIR}/cbam_neck_head/evaluation"

import os
from pathlib import Path

WEIGHTS_PT = f"{RUNS_DIR}/cbam_neck_head/weights/best.pt"
TEST_FILE = f"{DATASET_ROOT}/test_{SELECTED_RATIO}.txt"
VISUAL_DIR = f"{RUNS_DIR}/visual_eval_cbam_neck_head"
BENCHMARK_OUT = f"{RUNS_DIR}/cbam_neck_head/benchmark_latency.md"

if os.path.exists(WEIGHTS_PT):
    !python src/evaluation/benchmark_latency.py \
        --weights "{WEIGHTS_PT}" \
        --data configs/data/exp_2/data_colab_active.yaml \
        --imgsz 640 \
        --batch 1 \
        --warmup 10 \
        --runs 100 \
        --out "{BENCHMARK_OUT}"

    !python src/inference/visual_analysis.py \
        --weights "{WEIGHTS_PT}" \
        --test_file "{TEST_FILE}" \
        --output_dir "{VISUAL_DIR}" \
        --imgsz 640 \
        --max_images 30
else:
    print(f"⚠️ Chưa có {WEIGHTS_PT} để thực hiện đánh giá và trực quan hóa.")


#### Cell 3E_crack: Chạy Tiếp Tục (Resume) CBAM NECK + HEAD (Exp 3) Nếu Bị Crash 🔄


In [ ]:
import os
from pathlib import Path
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()
from ultralytics import YOLO

CHECKPOINT = f"{RUNS_DIR}/cbam_neck_head/weights/last.pt"
WEIGHTS_PT = f"{RUNS_DIR}/cbam_neck_head/weights/best.pt"
TEST_FILE = f"{DATASET_ROOT}/test_{SELECTED_RATIO}.txt"
VISUAL_DIR = f"{RUNS_DIR}/visual_eval_cbam_neck_head"
BENCHMARK_OUT = f"{RUNS_DIR}/cbam_neck_head/benchmark_latency.md"

if os.path.exists(CHECKPOINT):
    print(f"🔄 Đang tiếp tục huấn luyện CBAM NECK + HEAD (Exp 3) từ: {CHECKPOINT}")
    model = YOLO(CHECKPOINT)
    model.train(resume=True)
    !python src/evaluation/plot_comparison.py \
        --baseline_dir "{RUNS_DIR}/baseline" \
        --cbam_dir "{RUNS_DIR}/cbam_neck_head" \
        --output_dir "{RUNS_DIR}/cbam_neck_head/evaluation"

    if os.path.exists(WEIGHTS_PT):
        !python src/evaluation/benchmark_latency.py \
            --weights "{WEIGHTS_PT}" \
            --data configs/data/exp_2/data_colab_active.yaml \
            --imgsz 640 \
            --batch 1 \
            --warmup 10 \
            --runs 100 \
            --out "{BENCHMARK_OUT}"

        !python src/inference/visual_analysis.py \
            --weights "{WEIGHTS_PT}" \
            --test_file "{TEST_FILE}" \
            --output_dir "{VISUAL_DIR}" \
            --imgsz 640 \
            --max_images 30
else:
    print(f"❌ Không tìm thấy checkpoint: {CHECKPOINT}")


### Cell 3F: Huấn Luyện & Đánh Giá CBAM BACKBONE + HEAD 🌟


In [ ]:
!python src/training/train_cbam.py \
    --model configs/models/yolo11n_cbam_backbone_head.yaml \
    --data configs/data/exp_2/data_colab_active.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --save_period 10 \
    --aug_profile v2 \
    --project "{RUNS_DIR}" \
    --name cbam_backbone_head

!python src/evaluation/plot_comparison.py \
    --baseline_dir "{RUNS_DIR}/baseline" \
    --cbam_dir "{RUNS_DIR}/cbam_backbone_head" \
    --output_dir "{RUNS_DIR}/cbam_backbone_head/evaluation"

import os
from pathlib import Path

WEIGHTS_PT = f"{RUNS_DIR}/cbam_backbone_head/weights/best.pt"
TEST_FILE = f"{DATASET_ROOT}/test_{SELECTED_RATIO}.txt"
VISUAL_DIR = f"{RUNS_DIR}/visual_eval_cbam_backbone_head"
BENCHMARK_OUT = f"{RUNS_DIR}/cbam_backbone_head/benchmark_latency.md"

if os.path.exists(WEIGHTS_PT):
    !python src/evaluation/benchmark_latency.py \
        --weights "{WEIGHTS_PT}" \
        --data configs/data/exp_2/data_colab_active.yaml \
        --imgsz 640 \
        --batch 1 \
        --warmup 10 \
        --runs 100 \
        --out "{BENCHMARK_OUT}"

    !python src/inference/visual_analysis.py \
        --weights "{WEIGHTS_PT}" \
        --test_file "{TEST_FILE}" \
        --output_dir "{VISUAL_DIR}" \
        --imgsz 640 \
        --max_images 30
else:
    print(f"⚠️ Chưa có {WEIGHTS_PT} để thực hiện đánh giá và trực quan hóa.")


#### Cell 3F_crack: Chạy Tiếp Tục (Resume) CBAM BACKBONE + HEAD Nếu Bị Crash 🔄


In [ ]:
import os
from pathlib import Path
from src.models.cbam import register_cbam_to_ultralytics
register_cbam_to_ultralytics()
from ultralytics import YOLO

CHECKPOINT = f"{RUNS_DIR}/cbam_backbone_head/weights/last.pt"
WEIGHTS_PT = f"{RUNS_DIR}/cbam_backbone_head/weights/best.pt"
TEST_FILE = f"{DATASET_ROOT}/test_{SELECTED_RATIO}.txt"
VISUAL_DIR = f"{RUNS_DIR}/visual_eval_cbam_backbone_head"
BENCHMARK_OUT = f"{RUNS_DIR}/cbam_backbone_head/benchmark_latency.md"

if os.path.exists(CHECKPOINT):
    print(f"🔄 Đang tiếp tục huấn luyện CBAM BACKBONE + HEAD (Biến Thể Phối Hợp) từ: {CHECKPOINT}")
    model = YOLO(CHECKPOINT)
    model.train(resume=True)
    !python src/evaluation/plot_comparison.py \
        --baseline_dir "{RUNS_DIR}/baseline" \
        --cbam_dir "{RUNS_DIR}/cbam_backbone_head" \
        --output_dir "{RUNS_DIR}/cbam_backbone_head/evaluation"

    if os.path.exists(WEIGHTS_PT):
        !python src/evaluation/benchmark_latency.py \
            --weights "{WEIGHTS_PT}" \
            --data configs/data/exp_2/data_colab_active.yaml \
            --imgsz 640 \
            --batch 1 \
            --warmup 10 \
            --runs 100 \
            --out "{BENCHMARK_OUT}"

        !python src/inference/visual_analysis.py \
            --weights "{WEIGHTS_PT}" \
            --test_file "{TEST_FILE}" \
            --output_dir "{VISUAL_DIR}" \
            --imgsz 640 \
            --max_images 30
else:
    print(f"❌ Không tìm thấy checkpoint: {CHECKPOINT}")


## Bước 4️⃣: Đánh Giá & Vẽ Biểu Đồ So Sánh Tổng Hợp (Full Ablation Study)
> Cell này sẽ tự động phân tích tất cả các thư mục thí nghiệm (`baseline`, `cbam_backbone`, `cbam_neck`, `cbam_head`, `cbam_neck_head`, `cbam_backbone_head`), vẽ toàn bộ Learning Curves, Metrics Bar Comparison, và xuất bảng tổng kết Markdown xếp hạng tối ưu động vào thư mục `{RUNS_DIR}/full_evaluation`.

In [ ]:
!python src/evaluation/plot_comparison.py \
    --base_dir "{RUNS_DIR}" \
    --eval_dir "{RUNS_DIR}/full_evaluation"

print("\n📊 Bảng tổng kết Ablation Study:")
!cat "{RUNS_DIR}/full_evaluation/overall_ablation_summary.md"


## Bước 5️⃣: Đánh Giá Đối Sánh Trên Tập Test Lõi Chung (`test_core.txt`)
> **Mục đích:** Tập test lõi chung (535 ảnh) là tập kiểm tra bất biến giữa cả 3 tỷ lệ split (80/10/10, 70/15/15, 60/20/20).
> Đánh giá trên tập này giúp hội đồng thấy được hiệu năng khách quan của các mô hình mà không phụ thuộc vào kích thước tập test riêng.

In [ ]:
import os
import yaml
from pathlib import Path

if "SELECTED_RATIO" not in locals():
    SELECTED_RATIO = "70_15_15"

if "DATASET_ROOT" not in locals():
    extract_p = Path(f"/content/dataset_v2_{SELECTED_RATIO}")
    DATASET_ROOT = extract_p / "KLTN_prep_v2" if (extract_p / "KLTN_prep_v2").exists() else extract_p

if "RUNS_DIR" not in locals():
    if "SHARED_DRIVE" in locals() and SHARED_DRIVE:
        RUNS_DIR = f"{SHARED_DRIVE}/runs_exp2_{SELECTED_RATIO}"
    else:
        found_runs = list(Path("/content/drive/MyDrive").rglob(f"runs_exp2_{SELECTED_RATIO}"))
        RUNS_DIR = str(found_runs[0]) if found_runs else f"/content/runs_exp2_{SELECTED_RATIO}"

VARIANTS = ["baseline", "cbam_backbone", "cbam_neck", "cbam_head", "cbam_neck_head", "cbam_backbone_head"]
CORE_YAML = DATASET_ROOT / f"data_{SELECTED_RATIO}_core.yaml"

if not CORE_YAML.exists():
    base_yaml_path = DATASET_ROOT / f"data_{SELECTED_RATIO}.yaml"
    if base_yaml_path.exists():
        with open(base_yaml_path, "r", encoding="utf-8") as f:
            core_cfg = yaml.safe_load(f)
    else:
        core_cfg = {"path": str(DATASET_ROOT), "nc": 48}
    core_cfg["path"] = str(DATASET_ROOT)
    core_cfg["test"] = str(DATASET_ROOT / "test_core.txt")
    with open(CORE_YAML, "w", encoding="utf-8") as f:
        yaml.dump(core_cfg, f, allow_unicode=True, sort_keys=False)

print(f"🎯 Bat dau danh gia tren tap Test Loi: {CORE_YAML}")
print(f"📁 Thu muc checkpoint: {RUNS_DIR}")
for var_name in VARIANTS:
    weights_path = f"{RUNS_DIR}/{var_name}/weights/best.pt"
    if os.path.exists(weights_path):
        print(f"\n>>> Dang danh gia {var_name}...")
        !python src/evaluation/eval_core.py \
            --weights "{weights_path}" \
            --data "{CORE_YAML}" \
            --imgsz 640 --batch 16 \
            --out "{RUNS_DIR}/{var_name}/eval_test_core.json"
    else:
        print(f"⚠️ Bo qua {var_name} do chua co best.pt")
